In [9]:
mlflow.set_tracking_uri('http://3.111.55.134:5000')

In [13]:
import mlflow
# Set or create an experiment
mlflow.set_experiment("Exp 5 - ML Algos with HP Tuning.")

2026/07/17 22:46:10 INFO mlflow.tracking.fluent: Experiment with name 'Exp 5 - ML Algos with HP Tuning.' does not exist. Creating a new experiment.


<Experiment: artifact_location='s3://mlflow-buckets-areeba/10', creation_time=1784308564725, effective_trace_archival_retention=None, experiment_id='10', last_update_time=1784308564725, lifecycle_stage='active', name='Exp 5 - ML Algos with HP Tuning.', tags={}, trace_location=None, workspace='default'>

In [6]:
import optuna
import mlflow
import mlflow.sklearn
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from imblearn.over_sampling import SMOTE
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

In [7]:
df = pd.read_csv('dataset.csv').dropna()
df.shape

(36662, 2)

In [12]:
import mlflow.xgboost

In [14]:
# Step 1: Remap the class labels from [-1, 0, 1] to [2, 0, 1]
df['category'] = df['category'].map({-1: 2, 0: 0, 1: 1})

# Step 2: Remove rows where the target labels (category) are NaN
df = df.dropna(subset=['category'])

ngram_range = (1, 3)  # Trigram setting
max_features = 10000  # Set max_features to 1000 for TF-IDF

# Step 4: Train-test split before vectorization and resampling
X_train, X_test, y_train, y_test = train_test_split(df['clean_comment'], df['category'], test_size=0.2, random_state=42, stratify=df['category'])

# Step 2: Vectorization using TF-IDF, fit on training data only
vectorizer = TfidfVectorizer(ngram_range=ngram_range, max_features=max_features)
X_train_vec = vectorizer.fit_transform(X_train)  # Fit on training data
X_test_vec = vectorizer.transform(X_test)  # Transform test data

smote = SMOTE(random_state=42)
X_train_vec, y_train = smote.fit_resample(X_train_vec, y_train)

# Function to log results in MLflow
def log_mlflow(model_name, model, X_train, X_test, y_train, y_test):
    with mlflow.start_run():
        mlflow.set_tag("mlflow.runName", f"{model_name}_SMOTE_TFIDF_Trigrams")
        mlflow.set_tag("experiment_type", "algorithm_comparison")
        mlflow.log_param("algo_name", model_name)

        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        accuracy = accuracy_score(y_test, y_pred)
        mlflow.log_metric("accuracy", accuracy)

        classification_rep = classification_report(y_test, y_pred, output_dict=True)
        for label, metrics in classification_rep.items():
            if isinstance(metrics, dict):
                for metric, value in metrics.items():
                    mlflow.log_metric(f"{label}_{metric}", value)

        # Branch by model type to avoid the skops/untrusted-types issue
        if model_name.lower().startswith("xgb") or "xgboost" in model_name.lower():
            mlflow.xgboost.log_model(model, artifact_path=f"{model_name}_model")
        else:
            mlflow.sklearn.log_model(model, f"{model_name}_model")


def run_optuna_experiment():
    study = optuna.create_study(direction="maximize")
    study.optimize(objective_xgboost, n_trials=30)

    best_params = study.best_params
    best_model = XGBClassifier(
        n_estimators=best_params['n_estimators'],
        learning_rate=best_params['learning_rate'],
        max_depth=best_params['max_depth'],
        random_state=42
    )

    # log_mlflow will fit the model, log metrics, and save it correctly
    log_mlflow("XGBoost", best_model, X_train_vec, X_test_vec, y_train, y_test)

run_optuna_experiment()


[I 2026-07-17 22:46:31,130] A new study created in memory with name: no-name-bb286af3-b2bc-4a1a-9ac4-6c0b5ffba084
[I 2026-07-17 22:47:57,455] Trial 0 finished with value: 0.7218018652120359 and parameters: {'n_estimators': 266, 'learning_rate': 0.00035001142198475986, 'max_depth': 8}. Best is trial 0 with value: 0.7218018652120359.
[I 2026-07-17 22:49:56,495] Trial 1 finished with value: 0.8349463311631181 and parameters: {'n_estimators': 294, 'learning_rate': 0.014078778575943274, 'max_depth': 10}. Best is trial 1 with value: 0.8349463311631181.
[I 2026-07-17 22:51:56,488] Trial 2 finished with value: 0.7316558155903572 and parameters: {'n_estimators': 288, 'learning_rate': 0.0005468909343709667, 'max_depth': 9}. Best is trial 1 with value: 0.8349463311631181.
[I 2026-07-17 22:52:20,314] Trial 3 finished with value: 0.6869611120886856 and parameters: {'n_estimators': 123, 'learning_rate': 0.0025785304814348355, 'max_depth': 5}. Best is trial 1 with value: 0.8349463311631181.
[I 2026-0

🏃 View run XGBoost_SMOTE_TFIDF_Trigrams at: http://3.111.55.134:5000/#/experiments/10/runs/a667728ad0114d5aab52e40309d99452
🧪 View experiment at: http://3.111.55.134:5000/#/experiments/10
